# CNN Implementation

Companion code for the *Predictive Analytics Lab* (section 18). Run the cell below.

In [ ]:
# A small CNN for 8x8 sensor maps (Keras in Colab; a numpy fallback runs in the browser)
import numpy as np

rng = np.random.default_rng(0)
def make_maps(n):
    healthy = 0.25 + 0.03 * rng.normal(size=(n, 8, 8))   # low intensity, smooth
    faulty = 0.62 + 0.20 * rng.normal(size=(n, 8, 8))    # high intensity, noisy
    X = np.clip(np.r_[healthy, faulty], 0, 1); y = np.r_[np.zeros(n), np.ones(n)]
    return X[..., None], y

X, y = make_maps(300)
print("Dataset:", X.shape)
try:
    import tensorflow as tf                               # available in Google Colab
    model = tf.keras.Sequential([
        tf.keras.layers.Input((8, 8, 1)),
        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")])
    model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
    model.summary()
    model.fit(X, y, epochs=5, validation_split=0.2, verbose=2)
except ImportError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])     # a fixed (not learned) 3x3 kernel
    def features(m):
        m = m[..., 0]
        edges = np.mean([abs(np.sum(m[i:i+3, j:j+3] * lap)) for i in range(6) for j in range(6)])
        return [m.mean(), m.std(), edges]
    F = np.array([features(m) for m in X])
    print("TensorFlow is not available here, so we use convolution features + logistic regression.")
    print("CV accuracy:", cross_val_score(LogisticRegression(), F, y, cv=5).mean())